# 02 Interactive access -- Querying Deltalake and timetraveling

We will review some mechanisms from the previous session with the Delta table
generated in Apache Airflow.

The interesting table is **`s3://deltalake/server_metrics`** — written by your DAGs.

> **Tip:** if a cell says `Table not found`, your ingestion DAG hasn't run yet (or you have cleaned up and need to run it again). Go trigger it in Airflow first.

## Setup

In [ ]:
from deltalake import DeltaTable
import duckdb
import pandas as pd
import boto3
from botocore.exceptions import ClientError

# ── MinIO / S3 credentials ───────────────────────────────────────────────────
MINIO_ENDPOINT = "http://minio:9000"

storage_options = {
    "endpoint_url":               MINIO_ENDPOINT,
    "access_key_id":              "minioadmin",
    "secret_access_key":          "minioadmin",
    "region":                     "us-east-1",
    "allow_http":                 "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_ALLOW_HTTP":             "true",
}

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
)

SERVER_TABLE = "s3://deltalake/server_metrics"

print("✓ credentials configured")

In [ ]:
# ── DuckDB with S3 secret ────────────────────────────────────────────────────
con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute("""
CREATE OR REPLACE SECRET minio_secret (
    TYPE        s3,
    PROVIDER    config,
    ENDPOINT    'minio:9000',
    KEY_ID      'minioadmin',
    SECRET      'minioadmin',
    URL_STYLE   'path',
    USE_SSL     false
);
""")

print("✓ DuckDB ready — use delta_scan('s3://...') to query Delta tables")

## Inspect the Delta table

In [ ]:
# Table metadata and current version
dt = DeltaTable(SERVER_TABLE, storage_options=storage_options)

print(f"Current version : {dt.version()}")
print(f"\nSchema:")
for field in dt.schema().fields:
    print(f"  {field.name:<22} {field.type}")

In [ ]:
# Commit history — every Airflow DAG run that touched this table
history = dt.history(limit=20)
pd.DataFrame(history)[["version", "timestamp", "operation", "operationParameters"]]

---
## Query with DuckDB

`delta_scan()` reads the current version of the table directly from S3.

In [ ]:
# Row counts per datacenter
con.sql(f"""
SELECT
    datacenter,
    COUNT(*)                                     AS total_readings,
    ROUND(AVG(cpu_temp_c),  2)                   AS avg_cpu_temp,
    ROUND(AVG(memory_util_pct), 2)               AS avg_mem_util,
    SUM(CASE WHEN server_id IS NULL THEN 1 END)  AS null_server_ids
FROM delta_scan('{SERVER_TABLE}')
GROUP BY datacenter
ORDER BY datacenter
""").show()

In [ ]:
# Anomalous readings still in the table (there should be none after cleanup runs)
con.sql(f"""
SELECT *
FROM delta_scan('{SERVER_TABLE}')
WHERE
    cpu_temp_c      > 120  OR cpu_temp_c      < 0
    OR memory_util_pct > 100  OR memory_util_pct < 0
    OR power_draw_w    < 0
    OR server_id IS NULL
LIMIT 10
""").show()

## Time Travel

Change `version=` to any number shown in the history above to read
an older snapshot of the table — no backup needed, it's all in the
Delta transaction log.

In [ ]:
# Compare row counts across versions
current_version = dt.version()

print(f"{'Version':<10} {'Row count':<12} {'Operation':<20}")
print("-" * 44)

for h in reversed(dt.history()):
    v   = h["version"]
    op  = h.get("operation", "?")
    dt_v = DeltaTable(SERVER_TABLE, storage_options=storage_options, version=v)
    n   = dt_v.to_pyarrow_table().num_rows
    marker = " ← current" if v == current_version else ""
    print(f"v{v:<9} {n:<12,} {op:<20}{marker}")

In [ ]:
# ── Your turn ────────────────────────────────────────────────────────────────
# Pick a version BEFORE the cleanup ran and read the bad rows that were deleted.
#
# 1. Find the version number just before a DELETE operation in the history above.
# 2. Open that version and filter for the anomalous rows.

BEFORE_VERSION = 5   # <-- change this

dt_old  = DeltaTable(SERVER_TABLE, storage_options=storage_options, version=BEFORE_VERSION)
old_df  = dt_old.to_pyarrow_table().to_pandas()

bad = old_df[
    (old_df["cpu_temp_c"] > 120) | (old_df["cpu_temp_c"] < 0)
    | (old_df["memory_util_pct"] > 100) | (old_df["memory_util_pct"] < 0)
    | (old_df["power_draw_w"] < 0)
    | (old_df["server_id"].isna())
]

print(f"Rows in v{BEFORE_VERSION}: {len(old_df):,}  |  Anomalous: {len(bad):,}")
bad.head(10)